In [ ]:
import pandas as pd
import MySQLdb
from sqlalchemy import create_engine
import datetime as dt
import time as tm
import numpy as np
from pymongo import MongoClient
from math import radians, cos, sin,asin, sqrt
import pymysql
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100% !important; }</style>"))
pd.set_option('display.max_rows', 500)
pd.set_option('display.max_columns', 500)
pd.set_option('display.width', 100000)
import json
import urllib.request
import requests

import matplotlib.pyplot as plt
import seaborn as sns
plt.style.use('seaborn')

In [ ]:
import os
from config import *  # all DB / API handles come from environment variables, see config.py


db = client.get_database('telematics_db')






In [ ]:
gettime=lambda x: pd.Timestamp(x).timestamp()
getdate=lambda x: pd.datetime.fromtimestamp(x).date()
getDay=lambda x: pd.datetime.fromtimestamp(x).date()
dttoday=int(tm.time()-tm.time()%86400-19800)



In [ ]:
def loc_pings(l):
    vid=l[0]
    day_start=l[1]
    day_end=l[2]

    try:
        url=GPS_API_BASE+"/gps/vehicles?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(vid,int(day_start),int(day_end))
        print(url)
        
        data=json.load(urllib.request.urlopen(url))
        
        df=pd.DataFrame(data['data'][str(vid)])
        print("LOC PINGS A GAYI")
        
    except Exception as e:
        print("QUERY ISSUE AAYA HAI  " + str(e))
        df=pd.DataFrame()
    return df

In [ ]:
def hb_pings(l):
    vid=l[0]
    day_start=l[1]
    day_end=l[2]
    
    try:        
        url=GPS_API_BASE+"/heartbeat/list?vehicleIds={}&fromTime={}&toTime={}&sortOrder=&timeCondition=".format(vid,int(day_start),int(day_end))
        #hb_data = requests.get(url)

        print(url)

        hb_data=json.load(urllib.request.urlopen(url))
        hb=pd.DataFrame(hb_data['data'][str(vid)])
        print("HB AAYA")

    except Exception as e:

        print("HEARTBEAT ISSUE AAYA HAI  " + str(e))
        hb=pd.DataFrame()
        
    return hb

In [ ]:
def wire_issues(l):
    lc=loc_pings(l)
    hb=hb_pings(l)
    
    if len(hb)>1 & len(lc)>1:
        day_wire_issue=hb_lc(hb,lc)
    elif len(lc)>1 & len(hb)<1:
        day_wire_issue=lc_only(lc)
    else:
        print("DATA INSSUFICIENT")
        
        
    
    


In [ ]:
def lc_only(df):
    print(df)

In [ ]:
def hb_lc(hb,lc):    
    hb.sort_values('created',inplace=True)
    hb.created=h_beat.created/1000
    hb.created=h_beat.created.apply(int)
    lc.sort_values('time',inplace=True)
    
    if len(hb)<len(lc):
        df=pd.merge_asof(left=locs,right=h_beat,left_on='time',right_on='created')
    else:
        df=pd.merge_asof(left=h_beat,right=locs,left_on='created',right_on='time')
    df['shifted_chargeOn']=df['chargeOn'].shift(1)
    df.at[0,'shifted_chargeOn']=df['chargeOn'].iloc[0]
    
    
    
    return df


In [ ]:
vehicle_list=pd.read_sql("select vehicle_id from operator_vehicles where deboarding_time is null and vehicle_id=82000",con=mysql_vds)['vehicle_id'].tolist()

In [ ]:
def find_last_analysis(vid):
    data=pd.read_sql("select day from device_health where vehicle_id={}".format(vid),con=engine)
    last_day=data.day.max()
    start=gettime(last_day)
    end=gettime(dt.date.today())
    l=[]
    threshhold=86400
    while(start+threshhold<end):
        l.append([vid,int(start),int(start+threshhold)])
        start+=threshhold
    l.append([vid,int(start),int(end)])
    
    return l

In [ ]:
from multiprocessing.dummy import Pool as ThreadPool
pool = ThreadPool(10)
reuslts=pool.map(find_last_analysis,vehicle_list)
pool.close()
pool.join()

vid_start_end=[]
for i in reuslts:
    for k in i:
        vid_start_end.append(k)

In [ ]:
from multiprocessing.dummy import Pool as ThreadPool
pool = ThreadPool(10)
reuslts=pool.map(wire_issues,vid_start_end)
pool.close()
pool.join()


In [ ]:
hb=hb_pings([100002,int(tm.time()-886400),int(tm.time())])

In [ ]:
lc=loc_pings(([100002,int(tm.time()-886400),int(tm.time())]))

In [ ]:
hb.sort_values('created',ascending=True,inplace=True)
hb.created=hb.created/1000
hb.created=hb.created.apply(int)
hb.rename(index=str,columns={"created":"hb_created","updated":"hb_updated"},inplace=True)

lc.sort_values('time',ascending=True,inplace=True)
lc.rename(index=str,columns={"created":"lc_created"},inplace=True)


if len(hb)<=len(lc):
    df=pd.merge_asof(left=lc,right=hb,left_on='time',right_on='hb_created')
else:
    df=pd.merge_asof(left=hb,right=lc,left_on='hb_created',right_on='time')
    
df['shifted_chargeOn']=df['chargeOn'].shift(1)
df.at[0,'shifted_chargeOn']=df['chargeOn'].iloc[0]
    
df['location']

In [ ]:
df['location']

In [ ]:
df.replace(True,int(1),inplace=True)

In [ ]:
df.replace({True:1})

In [ ]:
a=pd.read_sql("select * from device_health ",con=engine)

In [ ]:
a.drop_duplicates(['day','vehicle_id'],inplace=True)

In [ ]:
data_pings=a[a['total_raw_pings']>0]

In [ ]:
plt.hist(a['total_raw_pings'],range(1,10000,10))

In [ ]:
data_pings.total_raw_pings.sum()

In [ ]:
data_pings[data_pings['day']==data_pings['day'].iloc[90000]]['total_distace_travelled'].sum()

In [ ]:
len(data_pings['vehicle_id'].unique())

In [ ]:
data_pings['day'].iloc[90000]

In [ ]:
data_pings[data_pings['day']=='2019-07-30'].total_raw_pings.sum()